In [ ]:
import numpy as np
import pandas as pd
from tqdm import tqdm
import argparse
import os

from utils.columns import *
from utils.imputation import impute_icustay_ids, fill_outliers, fill_stepwise, sample_and_hold, fixgaps, knn_impute
from utils.derived_features import calculate_onset, compute_pao2_fio2, compute_shock_index, compute_sofa, compute_sirs

### 1. build patient states for getting MDP

In [ ]:
class ChartEvents:
    """
    An object that manages a set of chart event tables, and 
    supports retrieving all chart events for a given ICU stay ID.
    """
    def __init__(self, ce_dfs, stay_id_col=C_ICUSTAYID):
        super().__init__()
        self.dfs = ce_dfs
        self.ranges = [(df[stay_id_col].min(), df[stay_id_col].max())
                       for df in ce_dfs]
        self.stay_id_col = stay_id_col
        
    def fetch(self, stay_id):
        results = []
        for df, (min_id, max_id) in zip(self.dfs, self.ranges):
            if stay_id >= min_id and stay_id <= max_id:
                results.append(df[df[self.stay_id_col] == stay_id])
        if results:
            return pd.concat(results)
        return None

def time_window(df, col, center_time, lower_window, upper_window):
    """
    Returns all rows in the given data frame whose timestamp in column 'col' fall
    between center_time - lower_window and center_time + upper_window.
    """
    return df[(df[col] >= center_time - lower_window) & (df[col] <= center_time + upper_window)]

def build_patient_states(chart_events, onset_data, demog, labU, MV, MV_procedure, winb4, winaft):
    """
    Builds the patient states dataframe without imputation. Returns two dataframes,
    one containing the patient states and the other containing the qstime
    dataframe, which maps ICU stay IDs to relevant timestamps in the stay.
    """    
    combined_data = []
    infection_times = []
    from_pe = 0
    
    for _, row in tqdm(onset_data.iterrows(), total=len(onset_data), desc='Building patient states'):
        qst = row[C_ONSET_TIME]  # flag for presumed infection
        icustayid = int(row[C_ICUSTAYID])
        assert qst > 0
        d1 = demog.loc[demog[C_ICUSTAYID] == icustayid, [C_AGE, C_DISCHTIME]].values.tolist()

        if d1[0][0] < 18:  # exclude younger than 18
            continue

        # Limit to a time period of -4h and +4h around the window of interest for sepsis3 definition)
        bounds = ((winb4 + 4) * 3600, (winaft + 4) * 3600)

        # Chart events, lab events, mech vent and extubated
        temp = time_window(chart_events.fetch(icustayid), C_CHARTTIME, qst, *bounds)
        temp2 = time_window(labU[labU[C_ICUSTAYID] == icustayid], C_CHARTTIME, qst, *bounds)
        temp3 = time_window(MV[MV[C_ICUSTAYID] == icustayid], C_CHARTTIME, qst, *bounds)
        if MV_procedure is not None:
            temp4 = time_window(MV_procedure[MV_procedure[C_ICUSTAYID] == icustayid], C_STARTTIME, qst, *bounds)
        else:
            temp4 = None
        
        # list of unique timestamps from all 3 sources / sorted in ascending order
        timesteps = sorted(pd.unique(pd.concat([
            temp[C_CHARTTIME], 
            temp2[C_CHARTTIME], 
            temp3[C_CHARTTIME]] + ([temp4[C_STARTTIME]] if temp4 is not None else []), ignore_index=True)))

        if len(timesteps) == 0:
            continue

        # Reformat each event, which may contain multiple items at the same timestep
        for i, timestep in enumerate(timesteps):

            # First three values: timestep, icustay ID, chart time
            item = {
                C_BLOC: i,
                C_ICUSTAYID: icustayid,
                C_TIMESTEP: timestep
            }
            
            # CHARTEVENTS: positions 4-31 (inclusive) are the 28 unique values
            # for different vitals, as stored in referenceMatrices['Refvitals']
            for _, event in temp[temp[C_CHARTTIME] == timestep].iterrows():
                if event[C_ITEMID] <= 0 or event[C_ITEMID] > len(CHART_FIELD_NAMES):
                    continue
               
                item[CHART_FIELD_NAMES[int(event[C_ITEMID] - 1)]] = event[C_VALUENUM]

            # LAB VALUES: positions 32-66 (inclusive) are the 35 unique values
            # for different lab tests, as stored in referenceMatrices['Reflabs']
            for _, event in temp2[temp2[C_CHARTTIME] == timestep].iterrows():
                if event[C_ITEMID] <= 0 or event[C_ITEMID] > len(LAB_FIELD_NAMES):
                    continue
                item[LAB_FIELD_NAMES[int(event[C_ITEMID] - 1)]] = event[C_VALUENUM]

            # MV: positions 67 and 68 store whether the measurement is relating
            # to being on a ventilator, and whether the patient was extubated
            matching_mv = (temp3[C_CHARTTIME] == timestep)
            if matching_mv.sum() > 0:
                if matching_mv.sum() > 1:
                    print(matching_mv.sum(), "MV items")
                event = temp3[matching_mv].iloc[0]
                item[C_MECHVENT] = event[C_MECHVENT]
                item[C_EXTUBATED] = event[C_EXTUBATED]
            
            # Second source of mechanical ventilation information
            if temp4 is not None and C_MECHVENT not in item and (temp4[C_STARTTIME] == timestep).sum() > 0:
                events = temp4[temp4[C_STARTTIME] == timestep]
                item[C_MECHVENT] = events[C_MECHVENT].any().astype(int)
                item[C_EXTUBATED] = events[C_EXTUBATED].any().astype(int)
                from_pe += 1
                
            combined_data.append(item)            

        infection_times.append({
            C_ICUSTAYID: icustayid,
            C_ONSET_TIME: qst,
            C_FIRST_TIMESTEP: timesteps[0], # first timestep
            C_LAST_TIMESTEP: timesteps[-1], # last timestep
            C_DISCHTIME: d1[0][1] # discharge time
        })
                
    print("Got {} items from procedure events".format(from_pe))
    state_df = pd.DataFrame(combined_data)
    qstime = pd.DataFrame(infection_times).set_index(C_ICUSTAYID)
    
    expected_columns = CHART_FIELD_NAMES + LAB_FIELD_NAMES
    for col in expected_columns:
        if col not in state_df.columns:
            print("Adding empty column '{}' (no data points)".format(col))
            state_df[col] = pd.NA
    return state_df, qstime

In [ ]:
PARENT_DIR = "./"
data_dir = os.path.join(PARENT_DIR)
out_dir = os.path.join(PARENT_DIR, 'intermediate_files', 'mdp')

In [ ]:
ce_paths = [os.path.join(data_dir, 'intermediate_files',path) for path in os.listdir(os.path.join(data_dir, 'intermediate_files')) if path.startswith("ce") and path.endswith(".csv")]


In [ ]:
ce_dfs = [pd.read_csv(path) for path in ce_paths]

In [ ]:
chart_events = ChartEvents(ce_dfs)

In [ ]:
print("Reading onset data...")
onset_data = pd.read_csv(os.path.join(data_dir, 'intermediate_files', 'sepsis_onset.csv'))

In [ ]:
print("Reading demog...")
demog = pd.read_csv(os.path.join(data_dir, 'intermediate_files', 'demog.csv'))

In [ ]:
print("Reading labs...")
labU = pd.concat([pd.read_csv(os.path.join(data_dir, 'intermediate_files', 'labs_ce.csv')),
                  pd.read_csv(os.path.join(data_dir, 'intermediate_files', 'labs_le.csv'))
                 ], ignore_index=True)

In [ ]:
print("Reading mechvent...")
MV = pd.read_csv(os.path.join(data_dir, 'intermediate_files', 'mechvent.csv'))
MV_procedure = pd.read_csv(os.path.join(data_dir, 'processed_files', 'mechvent_pe.csv'))

In [ ]:
WINDOW_BEFORE = 25
WINDOW_AFTER = 49

In [ ]:
state_df, qstime = build_patient_states(chart_events,
                                        onset_data,
                                        demog,
                                        labU,
                                        MV,
                                        MV_procedure,
                                        WINDOW_BEFORE,
                                        WINDOW_AFTER
                                        )

In [ ]:
print("Result: state_df contains {} rows, {} columns".format(len(state_df), len(state_df.columns)))

In [ ]:
print("Data availability:")
for col in sorted(state_df.columns):
    nan_fraction = pd.isna(state_df[col]).sum() / len(state_df)
    if nan_fraction > 0.0:
        print(col, nan_fraction)
print("")

In [ ]:
out_dir

In [ ]:
state_df.to_csv(os.path.join(out_dir, 'patient_states.csv'), index=False, float_format='%g')
qstime.to_csv(os.path.join(out_dir, 'qstime.csv'), float_format='%g')

### 2. Impute States for getting MDP

In [ ]:
df = pd.read_csv("./intermediate_files/mdp/patient_states.csv")
old_df = df.copy()

In [ ]:
def remove_outliers(df):
    # Transfer temperatures tagged as celsius but obviously fahrenheit
    wrong_unit_temps = (df[C_TEMP_C] > 90) & pd.isna(df[C_TEMP_F])
    df.loc[wrong_unit_temps, C_TEMP_F] = df.loc[wrong_unit_temps, C_TEMP_C]

    # Multiply FiO2 to be in percentage instead of fraction
    df.loc[df[C_FIO2_100] < 1, C_FIO2_100] = df.loc[df[C_FIO2_100] < 1, C_FIO2_100] * 100

    df = fill_outliers(df, {
        # Vitals
        C_WEIGHT: (None, 300),      # weight
        C_HR: (None, 250),      # HR
        C_SYSBP: (0, 300),         # BP
        C_MEANBP: (0, 200),
        C_DIABP: (0, 200),
        C_RR: (None, 80),      # RR
        C_SPO2: (None, 150),     # SpO2
        C_TEMP_C: (None, 90),      # temp
        C_FIO2_100: (20, None),      # FiO2
        C_FIO2_1: (None, 1.5),
        C_O2FLOW: (None, 70),      # O2 flow
        C_PEEP: (0, 40),         # PEEP
        C_TIDALVOLUME: (None, 1800),    # Tidal volume
        C_MINUTEVENTIL: (None, 50),      # Minute volume
        
        # Labs
        C_POTASSIUM: (1, 15),         # K+
        C_SODIUM: (95, 178),       # Na+
        C_CHLORIDE: (70, 150),       # Cl
        C_GLUCOSE: (1, 1000),       # Glc
        C_CREATININE: (None, 150),     # creatinine
        C_MAGNESIUM: (None, 10),      # Mg
        C_CALCIUM: (None, 20),      # Ca
        C_IONISED_CA: (None, 5),       # ionized Ca
        C_CO2_MEQL: (None, 120),     # CO2
        C_SGPT: (None, 10000),   # SGPT
        C_SGOT: (None, 10000),   # SGOT
        C_HB: (None, 20),      # hemoglobin
        C_HT: (None, 65),      # hematocrit
        C_WBC_COUNT: (None, 500),     # WBC
        C_PLATELETS_COUNT: (None, 2000),    # platelet
        C_INR: (None, 20),      # INR
        C_ARTERIAL_PH: (6.7, 8),        # pH
        C_PAO2: (None, 700),     # PO2
        C_PACO2: (None, 200),     # PCO2
        C_ARTERIAL_BE: (-50, None),     # base excess
        C_ARTERIAL_LACTATE: (None, 30),      # lactate
    })

    # Clamp SpO2 to 100 after removing outliers
    df[df[C_SPO2] > 100] = 100
    
    return df

In [ ]:
print("Remove outliers")
df = remove_outliers(df)

In [ ]:
def estimate_gcs(rass):
    """
    Estimates the Glasgow Coma Scale value from the value of
    the Richmond Agitation Sedation Scale.
    """
    if rass >= 0: return 15
    elif rass == -1: return 14
    elif rass == -2: return 12
    elif rass == -3: return 11
    elif rass == -4: return 6
    elif rass == -5: return 3
    return pd.NA

In [ ]:
print("Estimate GCS from RASS")
ii = pd.isna(df[C_GCS])
df.loc[ii, C_GCS] = df.loc[ii, C_RASS].apply(estimate_gcs)

In [ ]:
def convert_fio2_units(df):
    """Converts FiO2 Set (a.k.a. FiO2_1) to FiOS (a.k.a. FiO2_100)."""
    # approximate that FiO2 Set (col 24) ~= FiO2 (col 23) / 100 
    missing_fio2_set = pd.isna(df[C_FIO2_1]) & ~pd.isna(df[C_FIO2_100])
    df.loc[missing_fio2_set, C_FIO2_1] = df.loc[missing_fio2_set, C_FIO2_100] / 100
    missing_fio2 = ~pd.isna(df[C_FIO2_1]) & pd.isna(df[C_FIO2_100])
    df.loc[missing_fio2, C_FIO2_100] = df.loc[missing_fio2, C_FIO2_1] * 100
    return df

In [ ]:
def estimate_fio2(df):
    # First fill in missing values
    df = convert_fio2_units(df)
    
    sah_fio2 = {}
    for col in [C_INTERFACE, C_FIO2_100, C_O2FLOW]:    
        print("SAH on " + col)
        sah_fio2[col] = sample_and_hold(df[C_ICUSTAYID], df[C_TIMESTEP], df[col], SAH_HOLD_DURATION[col])
        print("Eliminated {:.1f}% of NA values".format((1 - pd.isna(sah_fio2[col]).sum() / pd.isna(df[col]).sum()) * 100))
    
    # NO FiO2, YES O2 flow, no interface OR cannula
    print('NO FiO2, YES O2 flow, no interface OR cannula ', end='')
    mask = (pd.isna(sah_fio2[C_FIO2_100]) & ~pd.isna(sah_fio2[C_O2FLOW]) & ((sah_fio2[C_INTERFACE] == 0) | (sah_fio2[C_INTERFACE] == 2)))
    print('({} rows) '.format(mask.sum()), end='')
    df.loc[mask, C_FIO2_100] = fill_stepwise(sah_fio2[C_O2FLOW].loc[mask], zip(*(
        [15, 12, 10, 8, 6, 5, 4, 3, 2, 1],
        [70, 62, 55, 50, 44, 40, 36, 32, 28, 24]
    )))
    print('[DONE]')

    # NO FiO2, NO O2 flow, no interface OR cannula
    print('NO FiO2, NO O2 flow, no interface OR cannula ', end='')
    mask = (pd.isna(sah_fio2[C_FIO2_100]) & pd.isna(sah_fio2[C_O2FLOW]) & ((sah_fio2[C_INTERFACE] == 0) | (sah_fio2[C_INTERFACE] == 2)))
    print('({} rows) '.format(mask.sum()), end='')

    df.loc[mask, C_FIO2_100] = 21
    print('[DONE]')

    # NO FiO2, YES O2 flow, face mask OR.... OR ventilator (assume it's face mask)
    print('NO FiO2, YES O2 flow, face mask OR.... OR ventilator (assume it\'s face mask) ', end='')
    mask = (pd.isna(sah_fio2[C_FIO2_100]) & ~pd.isna(sah_fio2[C_O2FLOW]) & (pd.isna(sah_fio2[C_INTERFACE]) | sah_fio2[C_INTERFACE].isin((1, 3, 4, 5, 6, 9, 10))))
    print('({} rows) '.format(mask.sum()), end='')

    df.loc[mask, C_FIO2_100] = fill_stepwise(sah_fio2[C_O2FLOW].loc[mask], zip(*(
        [15, 12, 10, 8, 6, 4],
        [75, 69, 66, 58, 40, 36]
    )))
    print('[DONE]')

    # NO FiO2, NO O2 flow, face mask OR ....OR ventilator
    print('NO FiO2, NO O2 flow, face mask OR ....OR ventilator ', end='')
    mask = (pd.isna(sah_fio2[C_FIO2_100]) & pd.isna(sah_fio2[C_O2FLOW]) & 
            (pd.isna(sah_fio2[C_INTERFACE]) | sah_fio2[C_INTERFACE].isin((1, 3, 4, 5, 6, 9, 10))))
    print('({} rows) '.format(mask.sum()), end='')

    df.loc[mask, C_FIO2_100] = pd.NA
    print('[DONE]')

    # NO FiO2, YES O2 flow, Non rebreather mask
    print('NO FiO2, YES O2 flow, Non rebreather mask ', end='')
    mask = (pd.isna(sah_fio2[C_FIO2_100]) & ~pd.isna(sah_fio2[C_O2FLOW]) & sah_fio2[C_INTERFACE] == 7)
    print('({} rows) '.format(mask.sum()), end='')

    df.loc[mask, C_FIO2_100] = fill_stepwise(sah_fio2[C_O2FLOW].loc[mask], zip(*(
        [9.99, 8, 6],
        [80, 70, 60]
    )), zip(*(
        [10, 15],
        [90, 100]
    )))
    print('[DONE]')

    # NO FiO2, NO O2 flow, NRM
    print('NO FiO2, NO O2 flow, NRM ', end='')
    mask = (pd.isna(sah_fio2[C_FIO2_100]) & pd.isna(sah_fio2[C_O2FLOW]) & sah_fio2[C_INTERFACE] == 7)
    print('({} rows) '.format(mask.sum()), end='')

    df.loc[mask, C_FIO2_100] = pd.NA
    print('[DONE]')

    # update again FiO2 columns
    df = convert_fio2_units(df)
    print('[DONE]')
    return df

In [ ]:
print("Estimate FiO2")
df = estimate_fio2(df)

In [ ]:
def estimate_vitals(df):
    # BP - if we have two values, we can impute the others using the definition of mean BP
    print('BP ', end='')
    ii = ~pd.isna(df.loc[:, C_SYSBP]) & ~pd.isna(df.loc[:, C_MEANBP]) & pd.isna(df.loc[:, C_DIABP])

    df.loc[ii, C_DIABP] = (3 * df.loc[ii, C_MEANBP] - df.loc[ii, C_SYSBP]) / 2
    ii = ~pd.isna(df.loc[:, C_SYSBP]) & ~pd.isna(df.loc[:, C_DIABP]) & pd.isna(df.loc[:, C_MEANBP])
    df.loc[ii, C_MEANBP] = (df.loc[ii, C_SYSBP] + 2 * df.loc[ii, C_DIABP]) / 3
    ii = ~pd.isna(df.loc[:, C_MEANBP]) & ~pd.isna(df.loc[:, C_DIABP]) & pd.isna(df.loc[:, C_SYSBP])
    df.loc[ii, C_SYSBP] = 3 * df.loc[ii, C_MEANBP] - 2 * df.loc[ii, C_DIABP]
    print('[DONE]')

    # TEMP
    # some values recorded in the wrong column
    print('TEMP ', end='')
    ii = (df.loc[:, C_TEMP_F] > 25) & (df.loc[:, C_TEMP_F] < 45)  # tempF close to 37deg??!
    df.loc[ii, C_TEMP_C] = df.loc[ii, C_TEMP_F]
    df.loc[ii, C_TEMP_F] = np.nan
    ii = (df.loc[:, C_TEMP_C] > 70)  # tempC > 70?!!! probably degF
    df.loc[ii, C_TEMP_F] = df.loc[ii, C_TEMP_C]
    df.loc[ii, C_TEMP_C] = np.nan
    
    ii = ~pd.isna(df.loc[:, C_TEMP_C]) & pd.isna(df.loc[:, C_TEMP_F])
    df.loc[ii, C_TEMP_F] = df.loc[ii, C_TEMP_C] * 1.8 + 32
    ii = ~pd.isna(df.loc[:, C_TEMP_F]) & pd.isna(df.loc[:, C_TEMP_C])
    df.loc[ii, C_TEMP_C] = (df.loc[ii, C_TEMP_F] - 32) / 1.8
    print('[DONE]')

    # Hb/Ht
    print('Hb/Ht ', end='')
    ii = ~pd.isna(df.loc[:, C_HB]) & pd.isna(df.loc[:, C_HT])
    df.loc[ii, C_HT] = (df.loc[ii, C_HB] * 2.862) + 1.216
    ii = ~pd.isna(df.loc[:, C_HT]) & pd.isna(df.loc[:, C_HB])
    df.loc[ii, C_HB] = (df.loc[ii, C_HT] - 1.216) / 2.862
    print('[DONE]')

    # BILI
    print('BILI ', end='')
    ii = ~pd.isna(df.loc[:, C_TOTAL_BILI]) & pd.isna(df.loc[:, C_DIRECT_BILI])
    df.loc[ii, C_DIRECT_BILI] = (df.loc[ii, C_TOTAL_BILI] * 0.6934) - 0.1752
    ii = ~pd.isna(df.loc[:, C_DIRECT_BILI]) & pd.isna(df.loc[:, C_TOTAL_BILI])
    df.loc[ii, C_TOTAL_BILI] = (df.loc[ii, C_DIRECT_BILI] + 0.1752) / 0.6934
    print('[DONE]')
    
    return df

In [ ]:
print("Estimate vitals")
df = estimate_vitals(df)

In [ ]:
print("Sample and hold")
sah_series = {C_BLOC: df[C_BLOC],
              C_ICUSTAYID: df[C_ICUSTAYID],
              C_TIMESTEP: df[C_TIMESTEP]
             }
for col in SAH_FIELD_NAMES:
    print("SAH on " + col)
    sah_series[col] = sample_and_hold(df[C_ICUSTAYID],
                                      df[C_TIMESTEP],
                                      df[col],
                                      SAH_HOLD_DURATION[col],
                                      col_name=col)
    print("Eliminated {:.1f}% of NA values".format((1 - pd.isna(sah_series[col]).sum() / pd.isna(df[col]).sum()) * 100))

df = pd.DataFrame(sah_series)
del sah_series

In [ ]:
print("Write")
df.to_csv("./intermediate_files/mdp/patient_states_filled.csv", index=False, float_format='%g')

### 3. build states and actions for getting MDP

In [ ]:
print("Reading states...")
df = pd.read_csv("./intermediate_files/mdp/patient_states_filled.csv")
qstime = pd.read_csv("./intermediate_files/mdp/qstime.csv")
qstime = qstime.set_index(C_ICUSTAYID, drop=True)

In [ ]:
print("Reading data files...")
demog = pd.read_csv("./intermediate_files/demog.csv")
inputpreadm = pd.read_csv("./processed_files/preadm_fluid.csv")
inputMV = pd.read_csv("./intermediate_files/fluid_align_with_snuh_mv.csv")
vasoMV = pd.read_csv("./intermediate_files/vaso_align_with_snuh_mv.csv")
note = pd.read_csv("./intermediate_files/note.csv")
UOpreadm = pd.read_csv("./processed_files/preadm_uo.csv")
UO = pd.read_csv("./processed_files/uo.csv")

In [ ]:
def build_states_and_actions(df, qstime, inputMV, inputCV, inputpreadm, vasoMV, vasoCV, demog, note, UOpreadm, UO, timestep_resolution, winb4, winaft, head=None, allowed_stays=None):
    """
    Performs two tasks: bins the data into time intervals defined by 
    timestep_resolution, and adds input and output information (vasopressors,
    fluids, and urine output).
    
    inputCV and vasoCV may be None if using MIMIC-IV (contains MetaVision only).
    """
    icustayidlist = np.unique(df[C_ICUSTAYID])
    icustayidlist = sorted(icustayidlist[~pd.isna(icustayidlist)])
    if allowed_stays is not None:
        old_count = len(icustayidlist)
        allowed_stays = set(allowed_stays)
        icustayidlist = [id for id in icustayidlist if id in allowed_stays]
        print("Filtered from {} to {} ICU stay ids".format(old_count, len(icustayidlist)))
    else:
        print("{} ICU stay IDs".format(len(icustayidlist)))
    if head:
        icustayidlist = icustayidlist[:head]

    combined_data = []
    bin_indexes = pd.Series(index=np.arange(len(df)), dtype=pd.Int64Dtype()) # Pointers corresponding to each row in df that point to the index of combined_data that this row went into
    
    # -52 until +28 = 80 hours in total
    total_duration = (winb4 + 3) + (winaft + 3)
    for icustayid in tqdm(icustayidlist, desc='Building states and actions'):

        # CHARTEVENTS AND LAB VALUES
        temp = df.loc[df[C_ICUSTAYID] == icustayid, :]  # subtable of interest
        beg = temp[C_TIMESTEP].iloc[0]  # timestamp of first record

        # IV FLUID STUFF
        input = inputMV.loc[inputMV[C_ICUSTAYID] == icustayid, :]  # subset of interest
        if inputCV is not None:
            input2 = inputCV.loc[inputCV[C_ICUSTAYID] == icustayid, :]  # subset of interest
        else:
            input2 = None
        startt = input[C_STARTTIME]  # start of all infusions and boluses
        endt = input[C_ENDTIME]  # end of all infusions and boluses
        rate = input[C_NORM_INFUSION_RATE]  # normalized rate of infusion (is NaN for boluses) || corrected for tonicity

        pread = inputpreadm.loc[inputpreadm[C_ICUSTAYID] == icustayid, C_INPUT_PREADM]  # preadmission volume
        if not pread.empty:  # store the value, if available
            totvol = pread.sum()
        else:
            totvol = 0  # if not documented: it's zero

        # compute volume of fluid given before start of record!!!
        t0 = 0
        t1 = beg
        # input from MV (4 ways to compute)
        infu = np.nansum(rate * (endt - startt) * ((endt <= t1) & (startt >= t0)) / 3600 +
                        rate * (endt - t0) * ((startt <= t0) & (endt <= t1) & (endt >= t0)) / 3600 +
                        rate * (t1 - startt) * ((startt >= t0) & (endt >= t1) & (startt <= t1)) / 3600 +
                        rate * (t1 - t0) * ((endt >= t1) & (startt <= t0)) / 3600)
        # all boluses received during this timestep, from inputMV (rate is always NaN for boluses)
        bolusMV = np.nansum(input.loc[pd.isna(input[C_RATE]) & (input[C_STARTTIME] >= t0) & (input[C_STARTTIME] <= t1), C_TEV])
        if input2 is not None:
            bolusCV = np.nansum(input2.loc[(input2[C_CHARTTIME] >= t0) & (input2[C_CHARTTIME] <= t1), C_TEV])
            bolus = bolusMV + bolusCV
        else:
            bolus = bolusMV
        totvol = np.nansum([totvol, infu, bolus])

        # VASOPRESSORS
        vaso1 = vasoMV.loc[vasoMV[C_ICUSTAYID] == icustayid, :]  # subset of interest
        if vasoCV is not None:
            vaso2 = vasoCV.loc[vasoCV[C_ICUSTAYID] == icustayid, :]  # subset of interest
        else:
            vaso2 = None
        startv = vaso1[C_STARTTIME]  # start of VP infusion
        endv = vaso1[C_ENDTIME]  # end of VP infusions
        ratev = vaso1[C_RATESTD]  # rate of VP infusion

        # note
        note1 = note.loc[note[C_ICUSTAYID] == icustayid, :]  # subset of interest

        # DEMOGRAPHICS / gender, age, elixhauser, re-admit, died in hosp?, died within
        # 48h of out_time (likely in ICU or soon after), died within 90d after admission?
        demog_row = demog[demog[C_ICUSTAYID] == icustayid].iloc[0]
        dem = {
            C_GENDER: demog_row[C_GENDER],
            C_AGE: demog_row[C_AGE],
            C_ELIXHAUSER: demog_row[C_ELIXHAUSER],
            C_RE_ADMISSION: demog_row[C_ADM_ORDER] > 1,
            C_DIED_IN_HOSP: demog_row[C_MORTA_HOSP],
            C_DIED_WITHIN_48H_OF_OUT_TIME: abs((demog_row[C_DOD]) - (demog_row[C_OUTTIME])) < (24 * 3600 * 2),
            C_MORTA_90: demog_row[C_MORTA_90],
            C_DELAY_END_OF_RECORD_AND_DISCHARGE_OR_DEATH: (qstime.loc[icustayid, C_DISCHTIME] - qstime.loc[icustayid, C_LAST_TIMESTEP]) / 3600
        }
        
        # URINE OUTPUT
        output = UO.loc[UO[C_ICUSTAYID] == icustayid, :]  # urine output for this icu stay
        pread = UOpreadm.loc[UOpreadm[C_ICUSTAYID] == icustayid, C_VALUE]  # preadmission UO
        if not pread.empty:  # store the value, if available
            UOtot = pread.sum()
        else:
            UOtot = 0
        # adding the volume of urine produced before start of recording! this could be
        # during the ICU stay but before the sepsis window
        UOnow = np.sum(output.loc[(output[C_CHARTTIME] >= t0) & (output[C_CHARTTIME] <= t1), C_VALUE])  # t0 and t1 defined above
        UOtot += UOnow

        for j in np.arange(0, total_duration, timestep_resolution):
            t0 = 3600 * j + beg  # left limit of time window
            t1 = 3600 * (j + timestep_resolution) + beg  # right limit of time window
            value = temp.loc[(temp[C_TIMESTEP] >= t0) & (temp[C_TIMESTEP] <= t1), :]  # index of items in this time period
            if len(value) == 0:
                continue
                
            # ICUSTAY_ID, OUTCOMES, DEMOGRAPHICS
            item = {
                C_BLOC: (j / timestep_resolution) + 1,  # 'bloc' = timestep (1,2,3...)
                C_ICUSTAYID: icustayid,       # icustay_ID
                C_TIMESTEP: int(3600 * j + beg),  # t0 = lower limit of time window
            }
            # Columns 4-11: demographics and outcomes
            item.update(dem)

            # #####################   DISCUSS ADDING STUFF HERE / RANGE, MIN, MAX ETC   ################

            # Columns 12-76: mean of chart and lab values in window
            # Are there categorical items in here?
            item.update({col: value[col].mean(skipna=True) for col in SAH_FIELD_NAMES})
            # shock index = HR/SBP and P/F (not yet computed)

            # VASOPRESSORS
            # for CV: dose at timestamps.
            # for MV: 4 possibles cases, each one needing a different way to compute the dose of VP actually administered:
            # ----t0---start----end-----t1----
            # ----start---t0----end----t1----
            # -----t0---start---t1---end
            # ----start---t0----t1---end----

            # MV
            v = (((endv >= t0) & (endv <= t1)) | ((startv >= t0) & (endv <= t1)) | ((startv >= t0) & (startv <= t1)) | ((startv <= t0) & (endv >= t1)))
            if vaso2 is not None:
                v2 = vaso2.loc[(vaso2[C_CHARTTIME] >= t0) & (vaso2[C_CHARTTIME] <= t1), C_RATESTD]
            else:
                v2 = pd.Series([], dtype=np.float64)

            if not ratev.loc[v].empty or not v2.empty:
                all_vs = np.concatenate([ratev.loc[v].values, v2.values])
                all_vs = all_vs[~np.isnan(all_vs)]
                if all_vs.size > 0:
                    item[C_MEDIAN_DOSE_VASO] = np.nanmedian(all_vs)
                    item[C_MAX_DOSE_VASO] = np.nanmax(all_vs)

            # INPUT FLUID
            # input from MV (4 ways to compute)
            infu = np.nansum(rate * (endt - startt) * ((endt <= t1) & (startt >= t0)) / 3600 +
                            rate * (endt - t0) * ((startt <= t0) & (endt <= t1) & (endt >= t0)) / 3600 +
                            rate * (t1 - startt) * ((startt >= t0) & (endt >= t1) & (startt <= t1)) / 3600 +
                            rate * (t1 - t0) * ((endt >= t1) & (startt <= t0)) / 3600)
            # all boluses received during this timestep, from inputMV (need to check rate is NaN) and inputCV (simpler):
            bolusMV = np.nansum(input.loc[pd.isna(input[C_RATE]) & (input[C_STARTTIME] >= t0) & (input[C_STARTTIME] <= t1), C_TEV])
            if input2 is not None:
                bolusCV = np.nansum(input2.loc[(input2[C_CHARTTIME] >= t0) & (input2[C_CHARTTIME] <= t1), C_TEV])
                bolus = bolusMV + bolusCV
            else:
                bolus = bolusMV
            
            # sum fluid given
            totvol = np.nansum([totvol, infu, bolus])
            item[C_INPUT_TOTAL] = totvol # total fluid
            item[C_INPUT_STEP] = infu + bolus # fluid at this step

            # UO
            UOnow = np.nansum(output.loc[(output[C_CHARTTIME] >= t0) & (output[C_CHARTTIME] <= t1), C_VALUE])
            UOtot = np.nansum([UOtot, UOnow])
            item[C_OUTPUT_TOTAL] = UOtot  # total UO
            item[C_OUTPUT_STEP] = UOnow  # UO at this step

            # CUMULATED BALANCE
            item[C_CUMULATED_BALANCE] = totvol - UOtot  # cumulated balance

            # Clinical note
            text_note = note1.loc[(note1[C_CHARTTIME] >= t0) & (note1[C_CHARTTIME] <= t1), "text"].values
            if len(text_note) >0:
                text_note = "".join(text_note)
                item[C_TEXT] = text_note
            else:
                text_note = "None"
                item[C_TEXT] = text_note
            
            bin_indexes[value.index] = len(combined_data) # Point to this row
            combined_data.append(item)

    result = pd.DataFrame(combined_data)
    expected_columns = DEMOGRAPHICS_FIELD_NAMES + SAH_FIELD_NAMES + IO_FIELD_NAMES + COMPUTED_FIELD_NAMES
    for col in expected_columns:
        if col not in result.columns:
            print("Adding empty column '{}' (no data points)".format(col))
            result[col] = pd.NA

    mapping_df = pd.DataFrame({
        C_BLOC: df[C_BLOC],
        C_ICUSTAYID: df[C_ICUSTAYID],
        C_TIMESTEP: df[C_TIMESTEP],
        C_BIN_INDEX: bin_indexes
    })
    return result, mapping_df

In [ ]:
inputCV = None
vasoCV = None
RESOLUTION = 4.0 #per time sampling
WINDOW_BEFORE = 25
WINDOW_AFTER = 49

In [ ]:
result, mapping = build_states_and_actions(df,
                                           qstime,
                                           inputMV,
                                           inputCV,
                                           inputpreadm,
                                           vasoMV,
                                           vasoCV,
                                           demog,
                                           note,
                                           UOpreadm,
                                           UO,
                                           RESOLUTION,
                                           WINDOW_BEFORE,
                                           WINDOW_AFTER,
                                           head=None,
                                           allowed_stays=None
                                            )

In [ ]:
print("Writing to file")
result.to_csv("./intermediate_files/mdp/states_and_actions.csv", index=False, float_format='%g')
mapping.to_csv("./intermediate_files/mdp/bin_mapping.csv", index=False, float_format='%g')

### 4. impute states and actions for getting MDP

In [ ]:
df = pd.read_csv("./intermediate_files/mdp/states_and_actions.csv")
old_df = df.copy()

In [ ]:
def correct_features(df):
    # CORRECT GENDER
    df[C_GENDER] = df[C_GENDER] - 1

    # CORRECT AGE > 200 yo
    ii = df[C_AGE] > 150
    df.loc[ii, C_AGE] = 91

    # FIX MECHVENT
    df.loc[pd.isna(df[C_MECHVENT]), C_MECHVENT] = 0
    df.loc[df[C_MECHVENT] > 0, C_MECHVENT] = 1

    # FIX Elixhauser missing values
    df.loc[pd.isna(df[C_ELIXHAUSER]), C_ELIXHAUSER] = np.nanmedian(df[C_ELIXHAUSER])  # use the median value / only a few missing data points

    # vasopressors / no NAN
    df.loc[pd.isna(df[C_MEDIAN_DOSE_VASO]), C_MEDIAN_DOSE_VASO] = 0
    df.loc[pd.isna(df[C_MAX_DOSE_VASO]), C_MAX_DOSE_VASO] = 0
    
    return df    

In [ ]:
print("Correcting features")
df = correct_features(df)

In [ ]:
print("Linear interpolation")
miss = pd.isna(df).sum() / len(df)
impute_columns = (miss > 0) & (miss < 0.05)  # less than 5# missingness
for col in df.loc[:,impute_columns].columns:
    print("Linear interpolation on", col)
    new_vals = fixgaps(df[col])
    df[col] = new_vals

In [ ]:
print("knn imputation")
knn_cols = CHART_FIELD_NAMES + LAB_FIELD_NAMES
df[knn_cols] = knn_impute(df[knn_cols], na_threshold=0.9, provenance=None)

In [ ]:
RESOLUTION = 4.0 #per time sampling

In [ ]:
print("Computing P/F")
df[C_PAO2_FIO2] = compute_pao2_fio2(df)
print("Computing shock index")
df[C_SHOCK_INDEX] = compute_shock_index(df)
print("Computing SOFA")
df[C_SOFA] = compute_sofa(df, timestep_resolution=RESOLUTION)
print("Computing SIRS")
df[C_SIRS] = compute_sirs(df)

In [ ]:
print("Write")
df.to_csv("./preprocessed_files/mimic_dataset.csv", index=False, float_format='%g')

### 5. Convert training data and compute conversion factors

In [ ]:
# Find sepsis cohort in the mimic dataset
import pandas as pd
mdp_data = pd.read_csv("./preprocessed_files/mimic_dataset.csv")
sepsis_cohort = pd.read_csv("./preprocessed_files/sepsis_cohort.csv")

- episode가 최소 7이상인 경우만 남김

In [ ]:
MIMICtable = mdp_data[mdp_data[C_ICUSTAYID].isin(sepsis_cohort[C_ICUSTAYID])].reset_index(drop=True)
MIMICtable.loc[MIMICtable['text'].isna(), 'text']='None'
print("original icustayid:",len(MIMICtable['icustayid'].unique()))
MIMICtable = MIMICtable.groupby(['icustayid']).filter(lambda x: len(x)>=7)
print("filtered icustayid:",len(MIMICtable['icustayid'].unique()))

In [ ]:
# headers I want to keep
dataheaders5 = ['bloc', 'icustayid', 'timestep', 'gender', 'age', 'elixhauser', 're_admission', 'died_in_hosp', 'died_within_48h_of_out_time', \
    'morta_90', 'delay_end_of_record_and_discharge_or_death', 'Weight_kg', 'GCS', 'HR', 'SysBP', 'MeanBP', 'DiaBP', 'RR', 'SpO2', 'Temp_C', 'FiO2_1', \
    'Potassium', 'Sodium', 'Chloride', 'Glucose', 'BUN', 'Creatinine', 'Magnesium', 'Calcium', 'Ionised_Ca', 'CO2_mEqL', 'SGOT', 'SGPT', 'Total_bili', 'Albumin', \
    'Hb', 'WBC_count', 'Platelets_count', 'PTT', 'PT', 'INR', 'Arterial_pH', 'paO2', 'paCO2', 'Arterial_BE', 'HCO3', 'Arterial_lactate', 'mechvent', 'Shock_Index', \
    'PaO2_FiO2', 'text', 'median_dose_vaso', 'max_dose_vaso', 'input_total', 'input_step', 'output_total', 'output_step', 'cumulated_balance', 'SOFA', 'SIRS']

In [ ]:
MIMICtable = MIMICtable[dataheaders5].copy()
MIMICtable['re_admission'] = MIMICtable['re_admission'].astype(int)

- IV fluid 이상치 제거 2000이상 기록이 있는 icustayid

In [ ]:
iv_outlier_id = MIMICtable.loc[MIMICtable['input_step']>=2000, 'icustayid'].unique()
MIMICtable = MIMICtable.loc[~MIMICtable['icustayid'].isin(iv_outlier_id)]
print("filtered icustayid:",len(MIMICtable['icustayid'].unique()))

- vaso 이상치 icustayid 제거

In [ ]:
vaso_outlier_id = MIMICtable.loc[MIMICtable['max_dose_vaso']>3, 'icustayid'].unique()
MIMICtable = MIMICtable.loc[~MIMICtable['icustayid'].isin(vaso_outlier_id)]
print("filtered icustayid:",len(MIMICtable['icustayid'].unique()))

- clinical note가 episode 당 최소 3개 이상 있는것만 남김

In [ ]:
df_note = MIMICtable[['bloc', 'icustayid', 'text']]
test = df_note.loc[df_note['text']!='None']
text_counts_per_id = test.groupby('icustayid')['text'].count()
ids_to_keep = text_counts_per_id[text_counts_per_id >= 3].index
print("final icustyaid: ",len(ids_to_keep))
MIMICtable = MIMICtable.loc[MIMICtable['icustayid'].isin(ids_to_keep)]

In [ ]:
print(MIMICtable.loc[MIMICtable['input_step']!=0, 'input_step'].mean())
print(MIMICtable.loc[MIMICtable['input_step']!=0, 'input_step'].std())
print(MIMICtable.loc[MIMICtable['input_step']!=0, 'input_step'].median())

In [ ]:
print(MIMICtable.loc[MIMICtable['max_dose_vaso']!=0, 'max_dose_vaso'].mean())
print(MIMICtable.loc[MIMICtable['max_dose_vaso']!=0, 'max_dose_vaso'].std())
print(MIMICtable.loc[MIMICtable['max_dose_vaso']!=0, 'max_dose_vaso'].median())

In [ ]:
print(MIMICtable['morta_90'].mean())
print(MIMICtable['morta_90'].std())
print(MIMICtable['morta_90'].median())

In [ ]:
for i in list(MIMICtable.columns):
    if i == 'text':
        pass
    else:
        try:
            print(i,"-mean :",MIMICtable[i].mean())
            print(i,"-std :",MIMICtable[i].std())
            print(i,"-median :",MIMICtable[i].median())
            print("---")
        except:
            print(i)

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
df_to_plot = pd.DataFrame(MIMICtable.loc[MIMICtable['input_step'] != 0, "input_step"])
plt.figure(figsize=(3, 4)) # Optionally adjust the figure size.
ax = df_to_plot.boxplot(showfliers=False) # Store the axes object in ax.
ax.yaxis.set_major_locator(mticker.MultipleLocator(100))
ax.yaxis.grid(True, which='major', linestyle='--', linewidth=0.5)
ax.set_title('Boxplot of Input 4-hourly (Non-zero values)')
ax.set_ylabel('Value')
plt.show()

In [ ]:
df_to_plot = pd.DataFrame(MIMICtable.loc[MIMICtable['max_dose_vaso'] != 0, "max_dose_vaso"])
print(df_to_plot['max_dose_vaso'].mean())
print(df_to_plot['max_dose_vaso'].std())
print(df_to_plot['max_dose_vaso'].median())
plt.figure(figsize=(3, 4)) # Optionally adjust the figure size.
ax = df_to_plot.boxplot(showfliers=False) # Store the axes object in ax.
ax.yaxis.set_major_locator(mticker.MultipleLocator(0.1))
ax.yaxis.grid(True, which='major', linestyle='--', linewidth=0.5)
ax.set_title('Boxplot of Input 4-hourly (Non-zero values)')
ax.set_ylabel('Value')
plt.show()

In [ ]:
print("IV의 0이 아닌 데이터 비율: ",(len(MIMICtable[MIMICtable['input_step'].values != 0]) / len(MIMICtable['input_step'].values)))
print("vasopressor의 0이 아닌 데이터 비율: ",(len(MIMICtable[MIMICtable['max_dose_vaso'].values != 0]) / len(MIMICtable['max_dose_vaso'].values)))

In [ ]:
qstime = pd.read_csv("./intermediate_files/mdp/qstime.csv")
qstime = qstime.set_index(C_ICUSTAYID, drop=True)
qstime = qstime.reset_index()
onset_data = qstime[['icustayid', 'onset_time']].drop_duplicates('icustayid')
MIMICtable = pd.merge(MIMICtable, onset_data, on='icustayid', how='left')

In [ ]:
target_id = MIMICtable['icustayid'].unique()
rng = np.random.default_rng(seed=42)
sampled_ids = rng.choice(target_id, size=700, replace=False)
sample_table = MIMICtable.loc[MIMICtable['icustayid'].isin(sampled_ids)]


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.ticker as ticker
from matplotlib.colors import Normalize
import matplotlib.cm as cm

# 1) Convert timestamps and compute relative times.
sample_table['charttime_dt'] = pd.to_datetime(sample_table['timestep'], unit='s')
if 'presumed_onset_dt' not in sample_table.columns:
    onset_times = (
        sample_table[['icustayid', 'onset_time']]
        .drop_duplicates('icustayid')
    )
    onset_times['presumed_onset_dt'] = pd.to_datetime(onset_times['onset_time'], unit='s')
    sample_table = sample_table.merge(
        onset_times[['icustayid', 'presumed_onset_dt']],
        on='icustayid', how='left'
    )
sample_table['relative_time_hours'] = (
    sample_table['charttime_dt'] - sample_table['presumed_onset_dt']
).dt.total_seconds() / 3600.0

# 2) Prepare the plot.
fig, ax = plt.subplots(figsize=(12, 6))

# 3) Filter the data, excluding zeros and missing values.
df = sample_table.dropna(subset=['relative_time_hours', 'max_dose_vaso'])
df = df[df['max_dose_vaso'] > 0]
x = df['relative_time_hours'].to_numpy()
y = df['max_dose_vaso'].to_numpy()
total_non_zero = len(x)

if total_non_zero > 0:
    # 4) Configure the bins.
    x_min, x_max = np.floor(x.min()), np.ceil(x.max())
    x_bins = np.arange(x_min, x_max + 1, 1)      # One-hour intervals.
    y_bins = np.arange(0, 0.5 + 0.025, 0.025)    # Bins from 0 to 0.5 with a step of 0.025.

    # 5) Compute the two-dimensional histogram and the actual proportions.
    counts, xedges, yedges = np.histogram2d(x, y, bins=[x_bins, y_bins])
    density = counts / total_non_zero          # Divide by the total sample count instead of using counts/40.

    # 6) Fix the color scale from 0 to the largest proportion.
    norm = Normalize(vmin=0, vmax=density.max())

    # 7) Draw the plot using pcolormesh.
    im = ax.pcolormesh(
        xedges, yedges, density.T,
        cmap=cm.Blues,
        shading='auto',
        norm=norm
    )
    cbar = fig.colorbar(im, ax=ax)
    cbar.ax.tick_params(labelsize=17)  # cbar tick fontsize
    cbar.set_label('Proportion', fontsize=22, labelpad=10)
else:
    ax.text(0.5, 0.5, 'No data to display',
            ha='center', va='center', transform=ax.transAxes)

# 8) Add reference lines, labels, and a title.
ax.tick_params(axis='x', labelsize=17)
ax.tick_params(axis='y', labelsize=17)
ax.axvline(0, color='r', linestyle='--',
           label='Presumed Onset')
ax.set_xlabel("Relative Time from Presumed Onset (Hours)", fontsize=22, labelpad=10)
ax.set_ylabel("Max Dose Vaso (mcg/kg/min)", fontsize=22, labelpad=10)
# ax.set_title("Density Heatmap (Proportion): Max Dose Vaso Over Relative Time")
ax.legend(fontsize=20)
ax.grid(False)

# 9) Configure axis limits and ticks.
ax.set_ylim(0, 0.5)
ax.xaxis.set_major_locator(ticker.MultipleLocator(4))
ax.set_xlim(max(-24, x_min - 1), min(48, x_max + 1))

plt.tight_layout()
plt.savefig("./plot/mimic4_vaso.png", dpi=600)
plt.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import matplotlib.ticker as ticker
from matplotlib.colors import Normalize
import matplotlib.cm as cm  # Use a colormap.
import math  # Import ceil for rounding up.

# Assume that sample_table or SNUHtable has already been loaded.
# Required columns: 'icustayid', 'timestep', 'onset_time', and 'input_step'.

# 1) Convert timestamps and compute relative times.
sample_table['charttime_dt'] = pd.to_datetime(sample_table['timestep'], unit='s')
if 'presumed_onset_dt' not in sample_table.columns:
    onset_times = (
        sample_table[['icustayid', 'onset_time']]
        .drop_duplicates('icustayid')
    )
    onset_times['presumed_onset_dt'] = pd.to_datetime(onset_times['onset_time'], unit='s')
    sample_table = sample_table.merge(
        onset_times[['icustayid', 'presumed_onset_dt']],
        on='icustayid', how='left'
    )
sample_table.dropna(subset=['charttime_dt','presumed_onset_dt'], inplace=True)
sample_table['relative_time_hours'] = (
    sample_table['charttime_dt'] - sample_table['presumed_onset_dt']
).dt.total_seconds() / 3600.0

# 2) Prepare the plot.
fig, ax = plt.subplots(figsize=(12, 6))

# 3) Filter the data to input_step > 0.
df = sample_table.dropna(subset=['relative_time_hours','input_step'])
df = df[df['input_step'] > 0]
x = df['relative_time_hours'].to_numpy()
y = df['input_step'].to_numpy()
total_non_zero = len(x)

if total_non_zero > 0:
    # 4) Configure the x-axis bins at one-hour intervals.
    x_min, x_max = np.floor(x.min()), np.ceil(x.max())
    x_bins = np.arange(x_min, x_max + 1, 1)

    # 5) Configure the y-axis bins using the 99th percentile.
    # 5) Configure the y-axis bins using the 99th percentile.
    y_min = 0
    q99 = np.quantile(y, 0.99)            # Only this part has changed.
    y_max = math.ceil(q99) if q99 > 0 else math.ceil(y.max())
    
    # Choose the bin step.
    step = 25 if y_max <= 500 else (50 if y_max <= 1000 else 100)
    y_bins = np.arange(y_min, y_max + step, step)

    # 6) Compute the two-dimensional histogram and convert counts to proportions.
    counts, xedges, yedges = np.histogram2d(x, y, bins=[x_bins, y_bins])
    density = counts / total_non_zero

    # 7) Fix the color scale from 0 to the global maximum.
    norm = Normalize(vmin=0, vmax=density.max())

    # 8) Draw the plot using pcolormesh.
    im = ax.pcolormesh(
        xedges, yedges, density.T,
        cmap=cm.Reds,
        shading='auto',
        norm=norm
    )
    cbar = fig.colorbar(im, ax=ax)
    cbar.ax.tick_params(labelsize=17)  # cbar tick fontsize
    cbar.set_label('Proportion', fontsize=22, labelpad=10)
else:
    ax.text(
        0.5, 0.5, 'No data to display',
        ha='center', va='center',
        transform=ax.transAxes
    )

# 9) Draw the presumed-onset reference line.
# 10) Configure labels and the title.
ax.tick_params(axis='x', labelsize=17)
ax.tick_params(axis='y', labelsize=17)
ax.axvline(0, color='r', linestyle='--',
           label='Presumed Onset')
ax.set_xlabel("Relative Time from Presumed Onset (Hours)", fontsize=22, labelpad=10)
ax.set_ylabel("IV Fluid Input (ml)", fontsize=22, labelpad=10)
ax.legend(fontsize=20)
ax.grid(False)

# 11) Configure axis limits and ticks.
ax.set_xlim(max(-24, x_min - 1), min(48, x_max + 1))
ax.set_ylim(y_min, y_max)
ax.xaxis.set_major_locator(ticker.MultipleLocator(4))

plt.tight_layout()
plt.savefig("./plot/mimic4_iv.png", dpi=600)
plt.show()

# 19.5 analyze clinical note

In [ ]:
df_note = MIMICtable[['bloc', 'icustayid', 'text']]
df_note['category'] = df_note['text'].str.extract(r'\[(.*?)\]', expand=False)
df_note['category'] = df_note['category'].fillna('None')
df_note.loc[df_note['category']=='radiology','category']='Radiology'
df_note.loc[df_note['category']=='dicharge summary','category']='Discharge summary'

In [ ]:
import seaborn as sns
# Exclude the 'Unknown' category.
# Create a new DataFrame containing only rows whose 'category' is not 'Unknown'.
df_filtered = df_note[df_note['category'] != 'None'].copy() # Use .copy() to create an independent DataFrame.

# Recalculate category ordering from the filtered data.
# Use value_counts() on the filtered data to determine the bar order.
filtered_counts = df_filtered['category'].value_counts()
print("--- 'Unknown' 제외 후 카테고리별 텍스트 수 ---")
print(filtered_counts)
print("-" * 30)


# Draw the plot using the updated data and order.
plt.figure(figsize=(10, 7))

# data=df_filtered uses the data after excluding 'Unknown'.
# order=filtered_counts.index uses the ordering calculated from the filtered data.
ax_v = sns.countplot(data=df_filtered,
                     x='category',
                     order=filtered_counts.index, # Apply the ordering of the filtered data.
                     palette='pastel')

# Increase the label and font sizes.
ax_v.set_xlabel('Category', fontsize=23, labelpad=10)
ax_v.set_ylabel('Number of clinical notes', fontsize=23, labelpad=10)

# Set the x-axis tick font size and rotation.
ax_v.tick_params(axis='x', labelsize=19)
ax_v.tick_params(axis='y', labelsize=19)

# Display counts above the bars using the existing behavior.
for patch in ax_v.patches:
    height = patch.get_height()
    x = patch.get_x() + patch.get_width() / 2
    ax_v.text(x,
              height + 0.1, # Leave a small gap above each bar for its count label.
              f'{int(height)}',
              ha='center',
              va='bottom', fontsize=15)

plt.tight_layout() # Adjust the layout.
plt.savefig("./plot/notes_count.png", dpi=600)
plt.show()

In [ ]:
# 3) Count observations in each category.
category_counts = df_note['category'].value_counts()

# -----------------------------------------------------
# 3.5) Group categories representing at most 2.6% into 'Etc'.
# -----------------------------------------------------
threshold_percent = 2.6  # Set the percentage threshold.
total_count = category_counts.sum() # Compute the total count.

# Convert the percentage threshold to a count; group categories at or below this count.
# Allow for floating-point rounding; very small differences are usually negligible.
threshold_count = total_count * (threshold_percent / 100.0)


# Retain categories whose counts are strictly greater than threshold_count.
main_categories = category_counts[category_counts > threshold_count]

# Find categories whose counts are at or below the threshold.
small_categories = category_counts[category_counts <= threshold_count]

# Check whether any categories need to be grouped.
if not small_categories.empty:
    # Sum the small-category counts into 'Etc'.
    etc_count = small_categories.sum()
    # Create a Pandas Series for the 'Etc' category.
    etc_series = pd.Series([etc_count], index=['Discharge summary'])
    # Combine the main categories with 'Etc'.
    # Use pd.concat with recent Pandas versions because Series.append is deprecated.
    plot_counts = pd.concat([main_categories, etc_series])
    # Optionally sort counts in descending order to inspect the results.
    plot_counts = plot_counts.sort_values(ascending=False)
else:
    # Use the original counts when there are no small categories.
    plot_counts = category_counts.sort_values(ascending=False) # Apply this step to keep the ordering consistent.



# -----------------------------------------------------
# 4) Draw the pie chart with 'Etc' and labels inside the chart.
# -----------------------------------------------------
plt.figure(figsize=(8, 8)) # Set the figure size.

# Use the Set3 colormap with the number of categories after grouping.
colors = cm.Pastel2(np.linspace(0, 1, len(plot_counts)))

# Use the existing function to display both percentages and counts.
def make_autopct(values):
    def my_autopct(pct):
        total = sum(values)
        val = int(round(pct*total/100.0))
        # For very small percentages, such as below 0.1%, display only the count or adjust the label.
        if pct < 0.1:
             return f'({val})' # Display only counts for very small values.
        return f'{pct:.1f}%' # For example: 15.5% (150).
    return my_autopct

# Use plt.pie() with plot_counts.index as the labels.
plt.pie(plot_counts.values,             # Size of each slice, represented by its count.
        labels=plot_counts.index,       # Labels for each slice, including 'Etc'.
        autopct=make_autopct(plot_counts.values), # Apply the function that displays percentages and counts.
        startangle=140,                 # Starting angle.
        colors=colors,                  # Apply the selected colors.
        pctdistance=0.8,               # Position the percentages; 0.6 is the default, and values closer to 1 move them outward.
        labeldistance=1.1,             # Position the labels; 1.1 is the default, and values above 1 move them outward.
        textprops={'fontsize': 19})      # Optionally adjust text properties, such as the font size.

# plt.title(f'Text Distribution by Category')
plt.axis('equal') # Use an equal aspect ratio so the pie chart is circular.
plt.tight_layout() # Adjust the layout.
plt.savefig("./plot/note_pi_distribution.png", dpi=600)
plt.show()

In [ ]:
test = df_note.loc[df_note['text']!='None']
counts_per_id  = test.groupby(['icustayid'])['text'].count()
counts_distribution = counts_per_id.value_counts().sort_index()

labels = list(dict(counts_distribution).keys())
counts = list(dict(counts_distribution).values())
x = np.arange(len(labels))

# Compute the median and third quartile.
median_val = counts_per_id.median()         # Median.
q3_val     = counts_per_id.quantile(0.75)   # 75th percentile, or third quartile (Q3).

# Find the x-coordinates corresponding to the median and Q3.
# Interpolate their positions if they do not occur exactly among the labels.
try:
    med_x = labels.index(median_val)
except ValueError:
    med_x = np.interp(median_val, labels, x)
try:
    q3_x = labels.index(q3_val)
except ValueError:
    q3_x = np.interp(q3_val, labels, x)

# Configure the style and figure.
plt.style.use('default')
fig, ax = plt.subplots(figsize=(9, 6))

# Extract colors from the colormap.
colors = plt.cm.GnBu(np.linspace(0.2, 0.9, len(labels)))

# Draw the bars.
bars = ax.bar(x, counts, color=colors, edgecolor='gray', linewidth=0.7, alpha=0.9)

# Configure ticks and labels.
ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=17)
ax.set_yticklabels(ax.get_yticks(), fontsize=17)
ax.set_xlabel('Text length', fontsize=20, labelpad=10)
ax.set_ylabel('Number of clinical notes', fontsize=20, labelpad=10)
# ax.set_title('Distribution of text per episode', fontsize=16, pad=15)

# Configure the grid and spines.
# ax.grid(axis='y', linestyle='--', linewidth=0.5, alpha=0.7)
# for spine in ['top', 'right']:
#     ax.spines[spine].set_visible(False)

# Display values above the bars.
# for bar in bars:
#     h = bar.get_height()
#     ax.text(
#         bar.get_x() + bar.get_width()/2,
#         h + max(counts)*0.01,
#         f'{h:,}',
#         ha='center', va='bottom',
#         fontsize=13
#     )

# Add vertical reference lines for the median and Q3.
ax.axvline(med_x, color='red', linestyle='--', linewidth=1.5,
           label=f'Median = {median_val:.1f}')
ax.axvline(q3_x, color='orange', linestyle=':', linewidth=1.5,
           label=f'Q3 = {q3_val:.1f}')

# Add the legend.
ax.legend(fontsize=17)
plt.tight_layout()
plt.savefig("./plot/notes_length.png", dpi=600)
plt.show()

In [ ]:
text_df = df_note.loc[df_note['text']!="None"]
text_df['text_length'] = text_df['text'].fillna('').str.len()

In [ ]:
# 1) Remove NaN values and extract the sequence-length series.
lengths = text_df['text_length'].dropna()

# 2) Compute the interquartile range (IQR).
Q1 = lengths.quantile(0.25)
Q3 = lengths.quantile(0.75)
IQR = Q3 - Q1

# 3) Define the outlier bounds.
lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

# 4) Remove outliers.
filtered = lengths[(lengths >= lower_bound) & (lengths <= upper_bound)]

# 5) Draw a box plot with a legend.
plt.figure(figsize=(8, 5), dpi=100)
bp = plt.boxplot(
    filtered,
    showfliers=False,
    showmeans=True,
    meanline=True,
    meanprops={'color': 'red'}
)

# Label the median and mean lines.
bp['means'][0].set_label('Mean')
bp['medians'][0].set_label('Median')

# Display the legend.
plt.legend(loc='best')

plt.title('MIMIC4 text distribution')
plt.xlabel('text')
plt.ylabel('length')
plt.grid(axis='x', linestyle='--', linewidth=0.5)
plt.show()

# 20. normalization

In [ ]:
# all 47 columns of interest + additional meta columns to easier associate trajectories with other patient auxiliary info (eg. notes)
colmeta = ['timestep', 'icustayid', "text"]  # Meta-data around patient stay
colbin = ['gender', 'max_dose_vaso', 're_admission']  # Binary features
# Patient features that will be z-normalize
colnorm = ['age', 'Weight_kg', 'GCS', 'HR', 'SysBP', 'MeanBP', 'DiaBP', 'RR', 'Temp_C', 'FiO2_1', 
        'Potassium', 'Sodium', 'Chloride', 'Glucose', 'Magnesium', 'Calcium', 'Ionised_Ca', 'CO2_mEqL',  'Hb', 
        'WBC_count', 'Platelets_count', 'PTT', 'PT', 'Arterial_pH', 'paO2', 'paCO2',
        'HCO3', 'Arterial_lactate', 'SOFA', 'SIRS', 'Shock_Index',
        'PaO2_FiO2', 'SpO2', 'BUN', 'Creatinine', 'SGOT', 'SGPT', 'Total_bili', 'Albumin', 'INR',
        'input_step', 'output_step']

In [ ]:
MIMICtable

In [ ]:
import joblib

# find patients who died in ICU during data collection period
icustayidlist = MIMICtable['icustayid']
icuuniqueids = icustayidlist.unique() #list of unique icustayids from MIMIC

MIMICraw = MIMICtable[colmeta + colbin + colnorm]
MIMICraw = MIMICraw.values  # RAW values

from sklearn.preprocessing import StandardScaler, MinMaxScaler
scaler = StandardScaler()
z_normalized_colnorm = scaler.fit_transform(MIMICtable[colnorm].values)

# Save to 'scaler.pkl'.
joblib.dump(scaler, './scaler/scaler_mimic4.pkl')


# Clip each value to the interval [-3, 3].
z_clipped = np.clip(z_normalized_colnorm, -3, 3)

# Apply min-max normalization to the interval [0, 1].
minmax_scaler = MinMaxScaler(feature_range=(0, 1))
z_minmax = minmax_scaler.fit_transform(z_clipped)


MIMICzs = np.hstack([MIMICtable[colmeta].values, 
                     MIMICtable[colbin].values, 
                     z_minmax
                    ])

MIMICzs[:,4] = np.log(MIMICzs[:, 4].astype(float) + 0.6)  # Scale MAX DOSE NORAD as in the original workflow; this feature is not used here.
# MIMICzs[:,46] = 2*MIMICzs[:,46]  # Increase weight of this variable ??????

In [ ]:
from scipy import stats
from collections import Counter
# # compute conversion factors using MIMIC data
# a = MIMICtable['input_step'].values  # IV fluid
# a = stats.rankdata(a[a>0])/len(a[a>0])  # Rank positive fluid values in ascending order; zero fluid remains action 1.
# iof = np.floor((a+0.2499999999)*4).astype(np.int64) #converts iv volume in 4 actions
# mask_pos = MIMICtable['input_step'] > 0 # location of non-zero fluid in big matrix
# io = np.ones(MIMICtable.shape[0]) # array of ones, by default     
# io[mask_pos] = iof + 1 # where more than zero fluid given: save actual action

# Counter(io)

In [ ]:
# Array of original values.
fluid = MIMICtable['input_step'].values
# Default action 1 represents no fluid.
io = np.ones_like(fluid, dtype=np.int64)
# 2 = 0 < fluid ≤ 60
io[(fluid > 0) & (fluid <= 50)] = 2
# 3 = 60 < fluid ≤ 120
io[(fluid > 50) & (fluid <= 180)] = 3
# 4 = 120 < fluid ≤ 320
io[(fluid > 180) & (fluid <= 320)] = 4
# 5 = fluid > 320
io[fluid > 320] = 5

Counter(io)

In [ ]:
# vc = MIMICtable['max_dose_vaso'].values.copy()
# vcr = stats.rankdata(vc[vc != 0])/len(vc[vc != 0])
# vcr = np.floor((vcr+0.249999999999)*4)
# vcr[vcr == 0] = 1
# vc[vc != 0] = vcr + 1
# vc[vc == 0] = 1

# Counter(vc)

In [ ]:
# Array of original values.
vaso = MIMICtable['max_dose_vaso'].values
# Default action 1 represents no fluid.
vc = np.ones_like(vaso, dtype=np.int64)
# 2 = 0 < fluid ≤ 60
vc[(vaso > 0) & (vaso <= 0.05)] = 2
# 3 = 60 < fluid ≤ 120
vc[(vaso > 0.05) & (vaso <= 0.10)] = 3
# 4 = 120 < fluid ≤ 320
vc[(vaso > 0.10) & (vaso <= 0.20)] = 4
# 5 = fluid > 320
vc[vaso > 0.20] = 5

Counter(vc)

In [ ]:
actions = (io-1)*5 + (vc-1)

In [ ]:
import matplotlib.patches as mpatches
unique_values, counts = np.unique(io, return_counts=True)

# Visualize the distribution with a bar chart.
plt.figure(figsize=(8, 6))
plt.bar(unique_values, counts, width=0.8, color="#840711")
plt.xlabel("IV Fluid Discrete Action", fontsize=17, labelpad=10)
plt.ylabel("Frequency", fontsize=17, labelpad=10)
plt.xticks(unique_values, fontsize=14)  # Display each unique value on the x-axis.
plt.yticks(fontsize=14)
plt.grid(alpha=0.2)
# Legend mapping for the categories.
legend_labels = [
    "1: 0 ml",
    "2: 0~50 ml",
    "3: 50~180 ml",
    "4: 180~320 ml",
    "5: >320 ml",
]
# Create dummy legend patches with the matching colors.
handles = [mpatches.Patch(facecolor="#840711", edgecolor="#840711", label=lab)
           for lab in legend_labels]

plt.legend(handles=handles, fontsize=14, frameon=True, loc="best")
plt.tight_layout()
plt.savefig("./plot/discrete_iv.png", dpi=600)
plt.show()

In [ ]:
MIMICtable["discret_IV"] = io
print(np.min(MIMICtable.loc[MIMICtable["discret_IV"]==1,"input_step"]), np.max(MIMICtable.loc[MIMICtable["discret_IV"]==1,"input_step"]), np.median(MIMICtable.loc[MIMICtable["discret_IV"]==1,"input_step"]))
print(np.min(MIMICtable.loc[MIMICtable["discret_IV"]==2,"input_step"]), np.max(MIMICtable.loc[MIMICtable["discret_IV"]==2,"input_step"]), np.median(MIMICtable.loc[MIMICtable["discret_IV"]==2,"input_step"]))
print(np.min(MIMICtable.loc[MIMICtable["discret_IV"]==3,"input_step"]), np.max(MIMICtable.loc[MIMICtable["discret_IV"]==3,"input_step"]), np.median(MIMICtable.loc[MIMICtable["discret_IV"]==3,"input_step"]))
print(np.min(MIMICtable.loc[MIMICtable["discret_IV"]==4,"input_step"]), np.max(MIMICtable.loc[MIMICtable["discret_IV"]==4,"input_step"]), np.median(MIMICtable.loc[MIMICtable["discret_IV"]==4,"input_step"]))
print(np.min(MIMICtable.loc[MIMICtable["discret_IV"]==5,"input_step"]), np.max(MIMICtable.loc[MIMICtable["discret_IV"]==5,"input_step"]), np.median(MIMICtable.loc[MIMICtable["discret_IV"]==5,"input_step"]))
MIMICtable.drop('discret_IV', axis=1, inplace=True)

In [ ]:
unique_values, counts = np.unique(vc, return_counts=True)

# Visualize the distribution with a bar chart.
plt.figure(figsize=(8, 6))
plt.bar(unique_values, counts, width=0.8, color="#08468B")
plt.xlabel("Vasopressor Discrete Action", fontsize=17, labelpad=10)
plt.ylabel("Frequency", fontsize=17, labelpad=10)
# plt.title("Distribution of Values")
plt.xticks(unique_values, fontsize=14)  # Display each unique value on the x-axis.
plt.yticks(fontsize=14)
plt.grid(alpha=0.2)
# Legend mapping for the categories.
legend_labels = [
    "1: 0 mcg/kg/min",
    "2: 0~0.05 mcg/kg/min",
    "3: 0.05~0.10 mcg/kg/min",
    "4: 0.10~0.20 mcg/kg/min",
    "5: >0.20 mcg/kg/min",
]
# Create dummy legend patches with the matching colors.
handles = [mpatches.Patch(facecolor="#08468B", edgecolor="#08468B", label=lab)
           for lab in legend_labels]

plt.legend(handles=handles, fontsize=14, frameon=True, loc="best")
plt.tight_layout()
plt.savefig("./plot/discrete_vaso.png", dpi=600)
plt.show()

In [ ]:
MIMICtable["discret_vaso"] = vc
print(np.min(MIMICtable.loc[MIMICtable["discret_vaso"]==1,"max_dose_vaso"]), np.max(MIMICtable.loc[MIMICtable["discret_vaso"]==1,"max_dose_vaso"]), np.median(MIMICtable.loc[MIMICtable["discret_vaso"]==1,"max_dose_vaso"]))
print(np.min(MIMICtable.loc[MIMICtable["discret_vaso"]==2,"max_dose_vaso"]), np.max(MIMICtable.loc[MIMICtable["discret_vaso"]==2,"max_dose_vaso"]), np.median(MIMICtable.loc[MIMICtable["discret_vaso"]==2,"max_dose_vaso"]))
print(np.min(MIMICtable.loc[MIMICtable["discret_vaso"]==3,"max_dose_vaso"]), np.max(MIMICtable.loc[MIMICtable["discret_vaso"]==3,"max_dose_vaso"]), np.median(MIMICtable.loc[MIMICtable["discret_vaso"]==3,"max_dose_vaso"]))
print(np.min(MIMICtable.loc[MIMICtable["discret_vaso"]==4,"max_dose_vaso"]), np.max(MIMICtable.loc[MIMICtable["discret_vaso"]==4,"max_dose_vaso"]), np.median(MIMICtable.loc[MIMICtable["discret_vaso"]==4,"max_dose_vaso"]))
print(np.min(MIMICtable.loc[MIMICtable["discret_vaso"]==5,"max_dose_vaso"]), np.max(MIMICtable.loc[MIMICtable["discret_vaso"]==5,"max_dose_vaso"]), np.median(MIMICtable.loc[MIMICtable["discret_vaso"]==5,"max_dose_vaso"]))
MIMICtable.drop('discret_vaso', axis=1, inplace=True)

In [ ]:
unique_values, counts = np.unique(actions, return_counts=True)

# Visualize the distribution with a bar chart.
plt.figure(figsize=(8, 4))
plt.bar(unique_values, counts, width=0.8)
plt.xlabel("Value")
plt.ylabel("Frequency")
plt.title("Distribution of Values")
plt.xticks(unique_values)  # Display each unique value on the x-axis.
plt.show()

In [ ]:
# 2) Create a DataFrame without column names.

columns = colmeta+colbin+colnorm
col_names = columns[6:]

df = pd.DataFrame(MIMICzs[:, 6:].astype(float), columns=col_names)

plt.figure(figsize=(8, 6))
df.boxplot(grid=False, patch_artist=True, showfliers=False)

# 4) Configure the plot.
plt.title("Box Plot with Mean & Std")
plt.xlabel("Columns")
plt.ylabel("Values")
plt.xticks(rotation=90)
plt.show()

In [ ]:
# Process MIMICzs into trajectory data for later use
MIMICzs             = pd.DataFrame(MIMICzs, columns=colmeta+colbin+colnorm)
meta_df             = pd.DataFrame(MIMICzs[colmeta].values, columns = colmeta)
ob_df               = pd.DataFrame(MIMICzs[colbin+colnorm].values, columns=colbin+colnorm)
ac_df               = pd.DataFrame(actions, columns=['action'])
raw_data_df         = MIMICtable.copy()
num_actions         = 25
outcome_key         = 'morta_90' #Should be either 'died_within_48h_of_out_time' or ''mortality_90d''
meta_cols           = meta_df.columns.tolist()
ob_cols             = ob_df.columns.tolist()

In [ ]:
import pyprind
raw_data_df['traj'] = (raw_data_df['bloc'] == 1).cumsum().values
meta_df['traj']     = (raw_data_df['bloc'] == 1).cumsum().values
ob_df['traj']       = (raw_data_df['bloc'] == 1).cumsum().values
ac_df['traj']       = (raw_data_df['bloc'] == 1).cumsum().values
trajectories        = raw_data_df['traj'].unique()
data = {}
data['meta_cols'] = meta_cols
data['obs_cols'] = ob_cols
data['traj'] = {}
print('Sepsis Cohort -- Making trajectory data')
bar = pyprind.ProgBar(len(trajectories))
for i in trajectories:
    bar.update()
    data['traj'][i] = {}
    data['traj'][i]['meta'] = meta_df[meta_df['traj']==i][meta_cols].values.T
    data['traj'][i]['obs'] = ob_df[ob_df['traj'] == i][ob_cols].values.T
    data['traj'][i]['actions'] = ac_df[ac_df['traj'] == i]['action'].values.astype(np.int32)
    data['traj'][i]['outcome'] = raw_data_df[raw_data_df['traj'] == i][outcome_key].values[0] 
    data['traj'][i]['rewards'] = np.zeros(len(data['traj'][i]['actions']))
    data['traj'][i]['rewards'][-1] = (1-2*data['traj'][i]['outcome'])


In [ ]:
print('Sepsis Cohort -- Making final output file')
col_names = ['traj', 'step']
col_names.extend(['m:'+ i for i in data['meta_cols']])
col_names.extend(['o:'+ i for i in data['obs_cols']])
col_names.append('a:action')
col_names.append('r:reward')
all_data = []
bar = pyprind.ProgBar(len(data['traj'].keys()))
for i in data['traj'].keys():
    bar.update()
    for ctr in range(data['traj'][i]['actions'].shape[0]):
        all_data.append([])
        all_data[-1].append(i)
        all_data[-1].append(ctr)
        for m_index in range(data['traj'][i]['meta'].shape[0]):
            all_data[-1].append(data['traj'][i]['meta'][m_index, ctr])
        for o_index in range(data['traj'][i]['obs'].shape[0]):
            all_data[-1].append(data['traj'][i]['obs'][o_index, ctr])
        all_data[-1].append(data['traj'][i]['actions'][ctr])
        all_data[-1].append(data['traj'][i]['rewards'][ctr])
df = pd.DataFrame(all_data, columns=col_names)
print("보상은 episode의 마지막에 생존하면 1을 부여하고 죽으면 -1을 부여")

In [ ]:
df.to_csv('./preprocessed_files/sepsis_final_data_withTimes.csv', index=False)

In [ ]:
raw_df              = pd.DataFrame(MIMICraw, columns=colmeta+colbin+colnorm)
meta_df             = raw_df[colmeta]
meta_df             = pd.DataFrame(meta_df.values, columns=colmeta)
ob_df               = raw_df[colbin+colnorm]
ob_df               = pd.DataFrame(ob_df.values, columns=colbin+colnorm)
ac_df               = pd.DataFrame(actions, columns=['action'])
raw_data_df         = MIMICtable.copy()
num_actions         = 25
outcome_key         = 'morta_90' #Should be either 'died_within_48h_of_out_time' or ''mortality_90d''
meta_cols           = meta_df.columns.tolist()
ob_cols             = ob_df.columns.tolist()
raw_data_df['traj'] = (raw_data_df['bloc'] == 1).cumsum().values
meta_df['traj']     = (raw_data_df['bloc'] == 1).cumsum().values
ob_df['traj']       = (raw_data_df['bloc'] == 1).cumsum().values
ac_df['traj']       = (raw_data_df['bloc'] == 1).cumsum().values
trajectories        = raw_data_df['traj'].unique()
data = {}
data['meta_cols'] = meta_cols
data['obs_cols'] = ob_cols
data['traj'] = {}
print('Sepsis Cohort -- Making RAW trajectory data')
bar = pyprind.ProgBar(len(trajectories))
for i in trajectories:
    bar.update()
    data['traj'][i] = {}
    data['traj'][i]['meta'] = meta_df[meta_df['traj']==i][meta_cols].values.T
    data['traj'][i]['obs'] = ob_df[ob_df['traj'] == i][ob_cols].values.T
    data['traj'][i]['actions'] = ac_df[ac_df['traj'] == i]['action'].values.astype(np.int32)
    data['traj'][i]['outcome'] = raw_data_df[raw_data_df['traj'] == i][outcome_key].values[0] 
    data['traj'][i]['rewards'] = np.zeros(len(data['traj'][i]['actions']))
    data['traj'][i]['rewards'][-1] = (1-2*data['traj'][i]['outcome'])

print('Sepsis Cohort -- Making final RAW output file')
col_names = ['traj', 'step']
col_names.extend(['m:'+ i for i in data['meta_cols']])
col_names.extend(['o:'+ i for i in data['obs_cols']])
col_names.append('a:action')
col_names.append('r:reward')
all_data = []
bar = pyprind.ProgBar(len(data['traj'].keys()))
for i in data['traj'].keys():
    bar.update()
    for ctr in range(data['traj'][i]['actions'].shape[0]):
        all_data.append([])
        all_data[-1].append(i)
        all_data[-1].append(ctr)
        for m_index in range(data['traj'][i]['meta'].shape[0]):
            all_data[-1].append(data['traj'][i]['meta'][m_index, ctr])            
        for o_index in range(data['traj'][i]['obs'].shape[0]):
            all_data[-1].append(data['traj'][i]['obs'][o_index, ctr])
        all_data[-1].append(data['traj'][i]['actions'][ctr])
        all_data[-1].append(data['traj'][i]['rewards'][ctr])
df = pd.DataFrame(all_data, columns=col_names)

In [ ]:
df.to_csv('./preprocessed_files/sepsis_final_data_RAW_withTimes.csv', index=False)

## appendix1 : scaling by MIMIC3 & SNUH SCALER

In [ ]:
import joblib

# find patients who died in ICU during data collection period
icustayidlist = MIMICtable['icustayid']
icuuniqueids = icustayidlist.unique() #list of unique icustayids from MIMIC

MIMICraw = MIMICtable[colmeta + colbin + colnorm]
MIMICraw = MIMICraw.values  # RAW values


scaler = joblib.load('./scaler/scaler_snuh.pkl') # mimic3 / snuh
z_normalized_colnorm = scaler.transform(MIMICtable[colnorm].values)



# Clip each value to the interval [-3, 3].
z_clipped = np.clip(z_normalized_colnorm, -3, 3)

# Apply min-max normalization to the interval [0, 1].
minmax_scaler = MinMaxScaler(feature_range=(0, 1))
z_minmax = minmax_scaler.fit_transform(z_clipped)


MIMICzs = np.hstack([MIMICtable[colmeta].values, 
                     MIMICtable[colbin].values, 
                     z_minmax
                    ])

MIMICzs[:,4] = np.log(MIMICzs[:, 4].astype(float) + 0.6)  # Scale MAX DOSE NORAD as in the original workflow; this feature is not used here.
# MIMICzs[:,46] = 2*MIMICzs[:,46]  # Increase weight of this variable ??????

In [ ]:
# Array of original values.
fluid = MIMICtable['input_step'].values
# Default action 1 represents no fluid.
io = np.ones_like(fluid, dtype=np.int64)
# 2 = 0 < fluid ≤ 60
io[(fluid > 0) & (fluid <= 50)] = 2
# 3 = 60 < fluid ≤ 120
io[(fluid > 50) & (fluid <= 180)] = 3
# 4 = 120 < fluid ≤ 320
io[(fluid > 180) & (fluid <= 320)] = 4
# 5 = fluid > 320
io[fluid > 320] = 5

# Array of original values.
vaso = MIMICtable['max_dose_vaso'].values
# Default action 1 represents no fluid.
vc = np.ones_like(vaso, dtype=np.int64)
# 2 = 0 < fluid ≤ 60
vc[(vaso > 0) & (vaso <= 0.05)] = 2
# 3 = 60 < fluid ≤ 120
vc[(vaso > 0.05) & (vaso <= 0.10)] = 3
# 4 = 120 < fluid ≤ 320
vc[(vaso > 0.10) & (vaso <= 0.20)] = 4
# 5 = fluid > 320
vc[vaso > 0.20] = 5

In [ ]:
actions = (io-1)*5 + (vc-1)

In [ ]:
# 2) Create a DataFrame without column names.

columns = colmeta+colbin+colnorm
col_names = columns[6:]

df = pd.DataFrame(MIMICzs[:, 6:].astype(float), columns=col_names)

plt.figure(figsize=(8, 6))
df.boxplot(grid=False, patch_artist=True, showfliers=False)

# 4) Configure the plot.
plt.title("Box Plot with Mean & Std")
plt.xlabel("Columns")
plt.ylabel("Values")
plt.xticks(rotation=90)
plt.show()

In [ ]:
# Process MIMICzs into trajectory data for later use
MIMICzs             = pd.DataFrame(MIMICzs, columns=colmeta+colbin+colnorm)
meta_df             = pd.DataFrame(MIMICzs[colmeta].values, columns = colmeta)
ob_df               = pd.DataFrame(MIMICzs[colbin+colnorm].values, columns=colbin+colnorm)
ac_df               = pd.DataFrame(actions, columns=['action'])
raw_data_df         = MIMICtable.copy()
num_actions         = 25
outcome_key         = 'morta_90' #Should be either 'died_within_48h_of_out_time' or ''mortality_90d''
meta_cols           = meta_df.columns.tolist()
ob_cols             = ob_df.columns.tolist()

raw_data_df['traj'] = (raw_data_df['bloc'] == 1).cumsum().values
meta_df['traj']     = (raw_data_df['bloc'] == 1).cumsum().values
ob_df['traj']       = (raw_data_df['bloc'] == 1).cumsum().values
ac_df['traj']       = (raw_data_df['bloc'] == 1).cumsum().values
trajectories        = raw_data_df['traj'].unique()
data = {}
data['meta_cols'] = meta_cols
data['obs_cols'] = ob_cols
data['traj'] = {}
print('Sepsis Cohort -- Making trajectory data')
bar = pyprind.ProgBar(len(trajectories))
for i in trajectories:
    bar.update()
    data['traj'][i] = {}
    data['traj'][i]['meta'] = meta_df[meta_df['traj']==i][meta_cols].values.T
    data['traj'][i]['obs'] = ob_df[ob_df['traj'] == i][ob_cols].values.T
    data['traj'][i]['actions'] = ac_df[ac_df['traj'] == i]['action'].values.astype(np.int32)
    data['traj'][i]['outcome'] = raw_data_df[raw_data_df['traj'] == i][outcome_key].values[0] 
    data['traj'][i]['rewards'] = np.zeros(len(data['traj'][i]['actions']))
    data['traj'][i]['rewards'][-1] = (1-2*data['traj'][i]['outcome'])

print('Sepsis Cohort -- Making final output file')
col_names = ['traj', 'step']
col_names.extend(['m:'+ i for i in data['meta_cols']])
col_names.extend(['o:'+ i for i in data['obs_cols']])
col_names.append('a:action')
col_names.append('r:reward')
all_data = []
bar = pyprind.ProgBar(len(data['traj'].keys()))
for i in data['traj'].keys():
    bar.update()
    for ctr in range(data['traj'][i]['actions'].shape[0]):
        all_data.append([])
        all_data[-1].append(i)
        all_data[-1].append(ctr)
        for m_index in range(data['traj'][i]['meta'].shape[0]):
            all_data[-1].append(data['traj'][i]['meta'][m_index, ctr])
        for o_index in range(data['traj'][i]['obs'].shape[0]):
            all_data[-1].append(data['traj'][i]['obs'][o_index, ctr])
        all_data[-1].append(data['traj'][i]['actions'][ctr])
        all_data[-1].append(data['traj'][i]['rewards'][ctr])
df = pd.DataFrame(all_data, columns=col_names)
print("보상은 episode의 마지막에 생존하면 1을 부여하고 죽으면 -1을 부여")

In [ ]:
df.to_csv('./preprocessed_files/sepsis_final_data_withTimes_scaling_by_snuh.csv', index=False)